In [ ]:
from tabnanny import verbose
from typing import Callable
import quantecon
from quantecon.markov import tauchen
import numpy as np
from numba import njit
import quantecon
import interpolation
from numba import guvectorize, float64
from dataclasses import dataclass
import time

## settings.py

In [ ]:
@njit
def maliar_grid(a_min, a_max, N, theta):
    a_grid = np.empty(N)
    for i in range(1,N+1):
        a_grid[i-1] = a_min + (a_max - a_min)*((i-1)/(N-1))**theta
        
    return a_grid

# 解く問題の設定を行う（パラメータ, グリッド, 効用関数や生産関数）
# モデルを変更する場合にはここを修正
class Setting:

    def __init__(self,
                beta=0.99,                       # 割引因子
                gamma=1,                         # 相対的リスク回避度(異時点間の代替弾力性の逆数)
                b=0,                             # 内生的な状態変数の最小値, 借入制約
                a_max=16,                        # 内生的な状態変数の最大値
                na=21,                           # 内生的な状態変数のグリッド数
                mu=0,                            # 外生変数のAR(1)過程の定数項
                rho=0.6,                         # 外生変数のAR(1)過程の慣性
                sigma=0.4,                       # 外生変数のAR(1)過程のショック項の標準偏差
                nz = 11,                         # 外生変数のグリッド数
                w=1.0,                           # 賃金の初期化
                r0 = 0.03,                       # 利子率の初期化
                alpha = 0.36,                    # 資本分配率
                delta = 0.05,                    # 固定資本減耗率
                lambdaPF = 1,                    # 政策関数の更新度
                tau = 0.1,                       # 資本所得課税率
                ):

        # パラメータを設定する
        self.r = r0
        self.beta = beta
        self.b = b
        self.gamma = gamma
        self.alpha = alpha
        self.delta = delta
        self.sigma = sigma
        self.rho = rho
        self.na = na
        self.nz = nz
        self.a_min = -b
        self.a_max = a_max
        self.lambdaPF = lambdaPF
        self.tau = tau

        # 外生変数の遷移確率とグリッドを設定する
        # mc = quantecon.markov.approximation.rouwenhorst(nz, rho, sigma, mu)
        mc = tauchen(nz, rho, sigma, mu, n_std = 2)
        self.Pz = np.array(mc.P)
        if mc.state_values is None:
            raise ValueError("mc.state_values is None, cannot apply np.exp.")
        self.z_grid = np.exp(mc.state_values)

        # 内生的な状態変数のグリッドを設定する
        # a_grid = np.linspace(-b, a_max, na)
        a_grid = maliar_grid(-b, a_max, na, theta = 2.0)
        self.a_grid = a_grid

        # 賃金を設定
        self.w = w

        # 政策関数の更新度を設定する
        self.lambdaPF = lambdaPF

        # CRRA型効用関数と限界効用を定義する
        gamma = self.gamma
        if gamma == 1:
            self.utility = np.log
            self.mutility: Callable[[float], float] = njit(lambda x: 1 / x)
        else:
            self.utility = njit(lambda x: x**(1-gamma) / (1 - gamma))
            self.mutility = njit(lambda x: x**(-gamma))


        # 政策関数の初期値を定義する
        z_grid = self.z_grid
        hfun_old = np.empty((len(a_grid), len(z_grid)))
        for i_a, a in enumerate(a_grid):
            for i_z, z in enumerate(z_grid):
                c_max = 0.5* ((1+r0) * a + z + b)
                hfun_old[i_a, i_z] = c_max
        self.hfun_old = hfun_old


        # 資本所得税の初期化
        self.Xi = 0.0

## policy_function.py

In [ ]:
# 線形補間を効率的に行うコード（interpolationパッケージのinterpでは正しい結果が得られなかった）
# I took this function from state-space Jacobian package
# @guvectorize(['void(float64[:], float64[:], float64[:], float64[:])'], '(n),(nq),(n)->(nq)')
@njit
def interpolate_y(x, xq, y, yq):
    """Efficient linear interpolation exploiting monotonicity.
    Complexity O(n+nq), so most efficient when x and xq have comparable number of points.
    Extrapolates linearly when xq out of domain of x.
    Parameters
    ----------
    x  : array (n), ascending data points
    xq : array (nq), ascending query points
    y  : array (n), data points
    Returns
    ----------
    yq : array (nq), interpolated points
    """
    nxq, nx = xq.shape[0], x.shape[0]

    xi = 0
    x_low = x[0]
    x_high = x[1]
    for xqi_cur in range(nxq):
        xq_cur = xq[xqi_cur]
        while xi < nx - 2:
            if x_high >= xq_cur:
                break
            xi += 1
            x_low = x_high
            x_high = x[xi + 1]

        xqpi_cur = (x_high - xq_cur) / (x_high - x_low)
        yq[xqi_cur] = xqpi_cur * y[xi] + (1 - xqpi_cur) * y[xi + 1]



# 特定のアルゴリズムを実行して政策関数を更新する関数を出力する
# FOCを変更する場合やアルゴリズムを変更する場合はここを修正

# 今期の状態変数について繰り返し記号はi, 来期の状態変数について繰り返し記号はj

def TimeIteration(hp: Setting): # hpはSettingクラスからつくられるインスタンス

    # インスタンスからローカル変数を定義する
    r, beta, b, mutility, w = hp.r, hp.beta, hp.b, hp.mutility, hp.w
    a_grid, z_grid, Pz, tau, Xi = hp.a_grid, hp.z_grid, hp.Pz, hp.tau, hp.Xi

    @njit
    def FOCs(c, a, z, i_z, hfun):

        # 制約式から次期の内生的な状態変数を計算する
        aprime =np.array([(1+(1-tau)*r) * a + w*z - c + Xi])

        expectation = 0
        for j_z in range(len(z_grid)):
            # 政策関数の候補を補間して次期の制御変数を計算する
            # cprime = interpolation.interp(a_grid, hfun[:, j_z], aprime)
            tmp = np.empty(1, dtype=np.float64)
            interpolate_y(a_grid, aprime, hfun[:, j_z], tmp)
            cprime = tmp[0]
            if cprime is None:
                raise ValueError("interp returned None, but a float is expected.")

            # オイラー方程式の右辺を計算する
            expectation += mutility(cprime) * Pz[i_z, j_z]

        rhs = max((1 + (1-tau)*r) * beta * expectation, mutility((1+(1-tau)*r) * a + w*z + b + Xi))

        FOC_diff = mutility(c) - rhs

        return FOC_diff


    @njit
    def UpdatePF(h_old):

        h_new = np.empty_like(h_old)
        for i_a, a in enumerate(a_grid):
            for i_z, z in enumerate(z_grid):
                # 第3引数は初期値
                c_star = quantecon.optimize.root_finding.brentq(FOCs, 1e-8, (1+(1-tau)*r) * a + w*z + b, args=(a, z, i_z, h_old)).root
                h_new[i_a, i_z] = c_star

        return h_new

    return UpdatePF





# メイン関数：特定のアルゴリズムでiterationを行い、問題を解く関数
# 基本的には変更する必要がない

def SolveProblem(hp,               # Settingクラスからつくられるインスタンス
                Algorithm,         # アルゴリズムを指定
                tol=1e-4,          # 許容繰り返し誤差
                max_iter=10000,     # iteration回数の最大値
                verbose=True,      # 進捗を表示するかどうか
                print_skip=25):    # 進捗を何回ごとに表示するか


    # インスタンスからローカル変数を定義する
    # R, beta, b, mutility = hp.R, hp.beta, hp.b, hp.mutility
    # a_grid, z_grid, Pz = hp.a_grid, hp.z_grid, hp.Pz
    lambdaPF = hp.lambdaPF
    hfun_old = hp.hfun_old

    # チェックのために外生変数のグリッドと遷移確率を表示する
    # print(f"About exogenous variables:")
    # print(f"grid is {z_grid}.")
    # print(f"Transition matrix is {Pz}.")



    # 政策関数を更新する関数を取得する
    UpdatePF = Algorithm(hp)

    # iterationを行い、問題を解く
    i = 0
    error = tol + 1

    while i < max_iter and error > tol:

        # 政策関数を更新する
        hfun_new_tilde = UpdatePF(hfun_old)

        # 古い政策関数と加重平均する
        hfun_new = lambdaPF*hfun_new_tilde + (1-lambdaPF)*hfun_old

        error = np.max(np.abs(hfun_new-hfun_old))
        i += 1
        if verbose and i % print_skip == 0:      # 進捗をprint_skip回ごとに表示する
            print(f"PF Error at iteration {i} is {error}.")
        hfun_old = hfun_new

    if i == max_iter:
        print("Failed to converge!")

    if verbose and i < max_iter:
        print(f"\nConverged in {i} iterations.")

    return hfun_new

## stationary_dist_fast.py

In [ ]:
# x0が存在する区間の左端のグリッドを見つける関数
@njit
def gridlookup2(x0, xgrid):
    nx = np.shape(xgrid)[0]
    ix = 0
    for jx in range(nx):
        if x0 <= xgrid[jx]:
            break
        ix += 1
    ix = min(max(1, ix), nx-1)
    return ix - 1


# 定常分布を求める関数
@njit
def solve_sd(mu0: np.ndarray, 
            na: int, 
            nz: int, 
            agrid: np.ndarray,
            Pz: np.ndarray,
            pfgrid: np.ndarray):
                        
    # 0. transition matrixを計算
    G = np.zeros((na*nz,na*nz))
    weight = np.zeros((na,nz))

    for iz in range(nz):
        for ia in range(na):
        
            # 政策関数 pfgrid[ia, iz] に合わせる
            aprime = pfgrid[ia, iz]

            
            if aprime < agrid[0]:
                weight[ia, iz] = 1.0
                jtilde = 0
            elif aprime > agrid[-1]:
                weight[ia, iz] = 0
                jtilde = na - 2
            else:
                # aprime が落ちる区間の左端のグリッドのインデックスを探す
                jtilde = gridlookup2(aprime, agrid)
                # 重み weight[ia, iz] に格納
                weight[ia, iz] = (agrid[jtilde+1] - aprime) / (agrid[jtilde+1] - agrid[jtilde])

            # スタックしたときの現在の状態のインデックスを計算
            i_s = iz * na + ia

            # 次期外生状態 jz でループ
            for jz in range(nz):

                # スタックしたときの次期の状態のインデックスを計算
                j_s  = jz * na + jtilde    

                # 遷移確率行列 G に代入
                #   - Pz[iz, jz] : 現在 iz から次期 jz への外生ショックの遷移確率
                G[i_s, j_s ] += weight[ia, iz]   * Pz[iz, jz]
                G[i_s, j_s + 1] += (1.0 - weight[ia, iz]) * Pz[iz, jz]

    
    # 1. iterationで定常分布を求める
    diffmu = 1.0e4
    mu1    = np.zeros((na, nz))

    # 分布の初期値をスタックしてベクトル化
    mu0 = mu0.T.flatten()

    while diffmu > 1e-6:

        # 1. 分布の更新 : dist_new = G' * dist
        mu1 = G.T @ mu0

        # 2. 収束判定
        diffmu = np.max(np.abs(mu1 - mu0))

        # 3. normalizeして次のループへ
        mu0 = mu1 / np.sum(mu1)
    
    sd = mu0.reshape(nz, na).T
    return sd

## equilibrium.py

In [ ]:
@dataclass
class Result:
    """ Result of equilibrium
    """
    r_star: float
    w_star: float
    K_star: float
    hfun_c: np.ndarray
    hfun_a: np.ndarray
    sd: np.ndarray
    converge_path: np.ndarray
    loop: int
    hp: Setting

def search_equilibrium(hp: Setting, lambdaR: float,DEBUG_MODE = False, tol = 1e-5) -> Result:
    """ Search equilibrium
    """


    converge_path = np.empty(0)

    diff = 1
    loop = 0
    while abs(diff) > tol:
        loop += 1

        # ローカル変数を定義
        alpha = hp.alpha #TODO: while文の外で良い
        delta = hp.delta
        r = hp.r
        na = hp.na
        nz = hp.nz
        tau = hp.tau

        # 1. 企業の利潤最大化条件から 総資本需要 K0d, 賃金 wage を求める
        Kd = ((r + delta) / alpha) ** (1 / (alpha - 1))
        wage = (1 - alpha) * (Kd ** alpha) # 賃金の情報を更新
        hp.w = wage
        hp.Xi = tau * r * Kd # 資本所得税の合計

        # 2. 個人の最適化問題を解いて 政策関数を求める
        hfun_c = SolveProblem(hp,TimeIteration, verbose=DEBUG_MODE)

        # 3. 定常分布を求める
        # hfun_c から 時期のアセット aの政策関数を求める
        hfun_aprime = np.empty((na, nz))
        a_mesh, z_mesh = np.meshgrid(hp.a_grid, hp.z_grid, indexing='ij') # ユニバーサル関数を使用するためのグリッドを生成
        hfun_aprime = (1+(1-tau)*r) * a_mesh + wage * z_mesh - hfun_c

        # 定常分布用のグリッドを用意
        # a_grid_sd = np.linspace(-hp.b, hp.a_grid[-1], len(hp.a_grid))

        # 定常分布の初期値を定義する
        # sd_grid = np.full((len(a_grid_sd), nz), 1.0 / (len(a_grid_sd) * nz)) # 各グリッドの初期値を均等に設定
        sd_grid = np.full((na, nz), 1.0 / (na * nz)) # 各グリッドの初期値を均等に設定
        # sd = sd_iteration(sd_grid, hfun_aprime, hp.a_grid, hp.Pz)
        # sd = solve_sd(sd_grid, len(a_grid_sd), len(hp.z_grid), a_grid_sd, hp.Pz, hfun_aprime)
        sd = solve_sd(sd_grid, na, nz, hp.a_grid, hp.Pz, hfun_aprime)

        # 4. 総資本供給と総資本需要の差分を計算
        # Amesh, _ = np.meshgrid(a_grid_sd, hp.z_grid, indexing='ij')
        Amesh, _ = np.meshgrid(hp.a_grid, hp.z_grid, indexing='ij')
        A = np.sum(Amesh * sd) # 総資本供給
        print("A: ", A)
        diff = (A - Kd)
        converge_path = np.append(converge_path, diff)
        if DEBUG_MODE:
            print("loop: ", loop)
            print("r: ", r, ", A: ", A, ", diff: ", diff)

        r = r -  lambdaR * diff
        hp.r = r # r0を更新


    hp.r = r + lambdaR * diff # 最後のループで更新されたrを使う
    return Result(r_star = hp.r, w_star = hp.w, K_star = Kd, 
                hfun_c = hfun_c, hfun_a = hfun_aprime, sd = sd, 
                converge_path = converge_path, loop = loop, hp = hp)

## main.py

In [ ]:
loops = 1
times = np.empty(loops)
r_stars = np.empty(loops)
k_star = np.empty(loops)
w_star = np.empty(loops)
for i in range(loops):
    start = time.time()
    hp = Setting(beta=0.96, gamma=3, rho=0.6, sigma=0.4, 
                alpha=0.36, delta=0.08, b=3, a_max=45,
                nz = 7, na = 50, r0 = 0.03, tau = 0.0) # brent法でエラーが出ないようにwの初期値を0以外に設定
    result = search_equilibrium(hp, lambdaR = 0.002, DEBUG_MODE= True)
    end = time.time()

    times[i] = (end - start)
    r_stars[i] = result.r_star
    k_star[i] = result.K_star
    w_star[i] = result.w_star

print(f"TIME: {np.mean(times)}({np.std(times)})")
print(f"r_star: {np.mean(result.r_star)}({np.std(result.r_star)})")
print(f"K_star: {np.mean(result.K_star)}({np.std(result.K_star)})")
print(f"w_star: {np.mean(result.w_star)}({np.std(result.w_star)})")


In [ ]:
import matplotlib.pyplot as plt

In [ ]:
# 横軸 i 縦軸result.hp.a_gridで2Dプロット
fig = plt.figure()
ax = fig.add_subplot(111)
ax.set_title("a_grid")
ax.set_xlabel("i")
plt.plot(np.linspace(0, 49,50), result.hp.a_grid)
plt.show()

In [ ]:
# result.sdを3Dプロット

In [ ]:
result.hfun_c[:,0]

In [ ]:
result.hp.hfun_old[:,0]